## Tabela: `workspace.silver.meta_alfabetizacao_uf`

| Item | Detalhe |
|------|---------|
| **Tabela** | `workspace.silver.meta_alfabetizacao_uf` |
| **Origem** | `workspace.bronze.meta_alfabetizacao_uf_raw` |
| **Descrição** | Tratamento e padronização dos dados de metas de alfabetização por UF, previamente ingeridos na camada Bronze. |
| **Camada** | Silver (tratamento e padronização) |
| **Colunas de controle** | `data_ingestao` (timestamp de carga), `fonte` (origem dos dados) |
| **Criado por** | suelen.ramos |
| **Data de criação** | 2025-01-20 |

### Chave de negócio
A combinação `ano + sigla_uf + rede` forma a chave única da tabela.

### Tratamentos realizados

- Padronização dos tipos de dados;
- Remoção de espaços;
- Padronização dos nomes das colunas;
- Preservação das informações originais;
- Preservação das colunas de controle da ingestão;
- Validação de duplicidades após a transformação.

In [0]:
%run ../../Monitoring/_common_monitoring

In [0]:
_m = iniciar_etapa()

In [0]:



spark.sql("""
CREATE OR REPLACE TABLE workspace.silver.meta_alfabetizacao_uf
USING DELTA
AS
SELECT
    CAST(ano AS INT) AS ano,
    TRIM(CAST(sigla_uf AS STRING)) AS sigla_uf,
    TRIM(CAST(rede AS STRING)) AS rede_codigo,

    CAST(taxa_alfabetizacao AS DOUBLE) AS taxa_alfabetizacao,

    CAST(meta_alfabetizacao_2024 AS DOUBLE) AS meta_alfabetizacao_2024,
    CAST(meta_alfabetizacao_2025 AS DOUBLE) AS meta_alfabetizacao_2025,
    CAST(meta_alfabetizacao_2026 AS DOUBLE) AS meta_alfabetizacao_2026,
    CAST(meta_alfabetizacao_2027 AS DOUBLE) AS meta_alfabetizacao_2027,
    CAST(meta_alfabetizacao_2028 AS DOUBLE) AS meta_alfabetizacao_2028,
    CAST(meta_alfabetizacao_2029 AS DOUBLE) AS meta_alfabetizacao_2029,
    CAST(meta_alfabetizacao_2030 AS DOUBLE) AS meta_alfabetizacao_2030,

    CAST(percentual_participacao AS DOUBLE) AS percentual_participacao,

    data_ingestao,
    fonte,
    lote_id

FROM workspace.bronze.meta_alfabetizacao_uf_raw

WHERE ano IS NOT NULL
  AND sigla_uf IS NOT NULL
  AND rede IS NOT NULL

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY ano, sigla_uf, rede
    ORDER BY data_ingestao DESC
) = 1;
""")

print("Tratamento da UF concluído na Silver!")

In [0]:
registrar_execucao("SILVER", "meta_uf", _m, row_count=contar_tabela("workspace.silver.meta_alfabetizacao_uf"))